# Fase 1 — Extracción y limpieza de texto (corpus RICSMUR)

Pipeline que toma los PDF descargados en la Fase 0 y extrae su texto con PyMuPDF, aplicando una limpieza progresiva en 8 pasos hasta dejar el corpus limpio y listo para el chunking (Fase 3):

1. **Escaneo inicial**: extrae el texto página a página y descarta documentos sin texto real (escaneados).
2. **Headers / footers**: detecta y elimina líneas repetidas entre páginas (cabeceras y pies de página).
3. **Números de página**: elimina líneas que son únicamente el número de página.
4. **Índices**: detecta y elimina páginas de tabla de contenidos mediante una puntuación heurística (numeración jerárquica, puntos/espacios líderes, palabras clave).
5. **Bibliografía**: detecta y elimina la sección de bibliografía/referencias bibliográficas.
6. **Autores**: detecta y elimina páginas de autoría, coordinación o comité científico.
7. **Referencias específicas**: elimina referencias y notas intratexto (citas numeradas, enlaces, DOIs...) que no se detectan a nivel de página completa.
8. **Limpieza final**: descarta páginas vacías y elimina la portada institucional.

Cada paso guarda su resultado intermedio en una subcarpeta propia (JSON) junto con un log de auditoría con las líneas eliminadas, lo que permite revisar exactamente qué se ha borrado en cada fase. La lista `target_files` fija los 20 documentos del Gold Standard que conforman el corpus final del trabajo.

In [1]:
# Instalación de PyMuPDF (extracción de texto de los PDF del corpus)
!pip install pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 77.4 MB/s eta 0:00:00


In [2]:
import fitz  # PyMuPDF: lectura eficiente de PDFs
import re    # Expresiones regulares para limpieza de texto
from collections import Counter  # Para contar repeticiones (headers/footers)

In [3]:
# Montar Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import zipfile
import os

# Ruta al zip en Drive
ZIP_PATH = "/content/drive/MyDrive/protocolos_medicos.zip"

# Carpeta destino en Colab
EXTRACT_PATH = "/content/protocolos_medicos"

# Crear carpeta si no existe
os.makedirs(EXTRACT_PATH, exist_ok=True)

# Descomprimir
with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
    zip_ref.extractall(EXTRACT_PATH)

print("ZIP descomprimido en:", EXTRACT_PATH)

ZIP descomprimido en: /content/protocolos_medicos


In [5]:
def extract_pages_text(pdf_path):
    """
    Extrae el texto de un PDF página a página.

    Se devuelve una lista de diccionarios con:
    - número de página
    - texto completo de esa página

    Trabajar página a página permite:
    - detectar headers/footers
    - eliminar portadas
    - mantener trazabilidad (muy importante en ámbito clínico)
    """
    doc = fitz.open(pdf_path)
    pages = []

    # Recorremos cada página del documento
    for page_num, page in enumerate(doc):
        # Extraemos el texto plano de la página
        text = page.get_text("text")

        pages.append({
            "page": page_num + 1,  # numeración humana (empieza en 1)
            "text": text.strip()   # eliminamos espacios sobrantes
        })

    return pages

In [6]:
def has_text(pages, min_chars=200):
    """
    Determina si un PDF contiene texto real o es un escaneo.

    Estrategia:
    - Sumamos el número total de caracteres extraídos
    - Si es menor que un umbral, asumimos que es un PDF escaneado

    Esto permite:
    - procesar primero PDFs nativos
    - dejar OCR como fase posterior
    """
    total_chars = sum(len(p["text"]) for p in pages)
    return total_chars >= min_chars

In [2]:
def clean_page(text):
    """
    Limpieza básica de una página: separa el texto en líneas, elimina
    espacios sobrantes y descarta las líneas vacías. Es el primer filtro,
    previo a las limpiezas específicas (headers/footers, números de
    página, índices, bibliografía...) aplicadas en las celdas siguientes.
    """

    lines = text.splitlines()
    cleaned = []
    total_lines = len(lines)

    for idx, line in enumerate(lines):
        line = line.strip()

        # 1. Eliminar líneas vacías
        if not line:
            continue

        # Si pasa todos los filtros, conservamos la línea
        cleaned.append(line)

    return "\n".join(cleaned)

In [ ]:
def process_pdf(pdf_path):
    """
    Procesa un único PDF: extrae el texto página a página, descarta el
    documento si no contiene texto real (escaneado) y aplica la limpieza
    básica de líneas, filtrando además las páginas demasiado cortas.
    """

    # 1. Extracción inicial
    pages = extract_pages_text(pdf_path)

    # 2. Verificar si es texto o escaneado
    if not has_text(pages):
        return {
            "status": "no_text",
            "pages": None
        }


    cleaned_pages = []

    # 8. Bucle de limpieza página a página
    for p in pages:
        final_text = clean_page(
            p["text"]
        )

        # Filtro de longitud mínima (evitar páginas vacías)
        if len(final_text) > 100:
            cleaned_pages.append({
                "page": p["page"],
                "text": final_text
            })

    return {
        "status": "text",
        "pages": cleaned_pages
    }

In [ ]:
import fitz
import re
from collections import Counter
import os
import json
import time


def normalize_text(text):
    """
    Normaliza una línea para comparar repeticiones entre páginas:
    minúsculas, sin dígitos y con espacios colapsados.
    """
    if not text: return ""
    text = text.lower()
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def detect_headers_footers(pages, top_n=10, bottom_n=10, threshold=0.7):
    """
    Detecta cabeceras y pies de página repetidos automáticamente.

    Estrategia:
    - Se analizan las primeras y últimas líneas de cada página
    - Si una línea aparece en un alto porcentaje de páginas,
      se considera header o footer

    Parámetros:
    - top_n: número de líneas superiores a analizar
    - bottom_n: número de líneas inferiores a analizar
    - threshold: porcentaje mínimo de repetición para considerar ruido
    """
    header_counter = Counter()
    footer_counter = Counter()
    header_original = {}
    footer_original = {}
    total_pages = len(pages)

    for p in pages:
        lines = p["text"].splitlines()
        header_lines = lines[:top_n]
        footer_lines = lines[-bottom_n:]

        for line in header_lines:
            line = line.strip()
            if not line: continue
            norm = normalize_text(line)
            if norm:
                header_counter[norm] += 1
                header_original.setdefault(norm, set()).add(line)

        for line in footer_lines:
            line = line.strip()
            if not line: continue
            norm = normalize_text(line)
            if norm:
                footer_counter[norm] += 1
                footer_original.setdefault(norm, set()).add(line)

    # Seleccionamos solo líneas muy repetidas (texto ORIGINAL)
    headers = {orig for norm, count in header_counter.items() if count / total_pages >= threshold for orig in header_original.get(norm, [])}
    footers = {orig for norm, count in footer_counter.items() if count / total_pages >= threshold for orig in footer_original.get(norm, [])}
    return headers, footers



In [10]:

def remove_headers_footers_with_log(text, headers, footers):
    """
    Elimina de un texto las líneas que coinciden con los headers/footers
    detectados, registrando en 'removed' las líneas eliminadas para el
    log de auditoría del pipeline.
    """
    lines = text.splitlines()
    cleaned = []
    removed = []  # líneas eliminadas, para el log de auditoría

    for line in lines:
        line_stripped = line.strip()

        if line_stripped in headers or line_stripped in footers:
            removed.append(line_stripped)  # se registra para el log
            continue

        cleaned.append(line)  # se conserva la línea original, con sus espacios

    return "\n".join(cleaned), removed

In [ ]:
def is_page_number_line(line, page_number, line_index, total_lines, top_zone=2, bottom_zone=3, tolerance=5):
    """
    Determina si una línea es, con alta probabilidad, un número de
    página: debe ser puramente numérica, próxima al número de página
    esperado, y aparecer en la zona de cabecera o de pie (no en el cuerpo).
    """
    # 1. La línea debe ser solo un número
    if not line.isdigit(): return False
    num = int(line)
    # 2. El número debe estar cerca del número de página esperado
    if abs(num - page_number) > tolerance: return False
    # 3. Debe estar en cabecera o pie
    if line_index < top_zone: return True
    if line_index >= total_lines - bottom_zone: return True
    return False

In [12]:
def clean_page_numbers_with_log(text, page_number):
    """
    Elimina los números de página reales detectados por
    is_page_number_line, registrando las líneas eliminadas en 'removed'.
    """
    lines = text.splitlines()
    cleaned = []
    removed = []  # líneas eliminadas, para el log de auditoría
    total_lines = len(lines)

    for idx, line in enumerate(lines):
        line = line.strip()

        if not line:
            continue

        # Eliminar números de página reales
        if is_page_number_line(
            line=line,
            page_number=page_number,
            line_index=idx,
            total_lines=total_lines
        ):
            removed.append(line)  # se registra para el log
            continue


        cleaned.append(line)

    return "\n".join(cleaned), removed

In [ ]:
import re
import math

def index_score(text):
    """
    Puntuación heurística (0-1) de cuánto se parece el texto de una
    página a un índice/tabla de contenidos: combina palabras clave,
    numeración jerárquica, líderes de puntos/espacios y penaliza páginas
    con alta densidad de dígitos (que suelen ser tablas reales, no índices).
    """
    text_lower = text.lower()
    lines = [l.strip() for l in text.splitlines() if l.strip()]

    if not lines:
        return 0.0

    # --- 1. Palabras clave ---
    keywords = [
        "índice", "indice", "tabla de contenidos", "sumario", "contenido",
        "anexos", "bibliografía", "referencias"
    ]
    has_keyword = any(k in text_lower for k in keywords)

    # --- 2. Patrones ---

    # Jerarquía numérica (1., 1.1, 1.2.3)
    hierarchical_pattern = re.compile(
        r'^(?:\d+\.[-\s]?|\d+(?:\.\d{1,2})+[.-]?)\s*$'
    )

    # Líderes clásicos
    dotted_pattern = re.compile(r'\.{3,}\s*\d+$')

    # Líderes por espacios
    space_leader_pattern = re.compile(r'\s{6,}\d+$')

    # Numeración romana genérica
    roman_pattern = re.compile(
        r'^(?:-?\s*)?(?:[IVXLCDM]+)[\.\-:]?(?:\s+\D+)?$'
    )

    # Listas tipo índice: "- TABLA X:"
    list_roman_pattern = re.compile(
        r'^-\s+\D+\b[IVXLCDM]+\b\s*:'
    )

    lines_ending_in_number = 0
    lines_with_dotted_leader = 0
    lines_with_space_leader = 0
    numeric_heavy_lines = 0
    hierarchical_numbered_lines = 0
    roman_lines = 0
    list_roman_lines = 0

    for line in lines:
        clean = re.sub(r'[.\s]+$', '', line)

        # 1. Termina en número
        if clean and clean[-1].isdigit():
            lines_ending_in_number += 1

        # 2. Líderes
        has_dots = bool(dotted_pattern.search(line))
        has_spaces = bool(space_leader_pattern.search(line))

        if has_dots:
            lines_with_dotted_leader += 1
        if has_spaces:
            lines_with_space_leader += 1

        # 3. Jerarquía numérica
        is_hierarchical = bool(hierarchical_pattern.match(line))
        if is_hierarchical:
            hierarchical_numbered_lines += 1

        # 4. Romanos
        is_roman = bool(roman_pattern.match(line))
        if is_roman:
            roman_lines += 1

        # 5. Listas romanas tipo índice
        is_list_roman = bool(list_roman_pattern.search(line))
        if is_list_roman:
            list_roman_lines += 1

        # 6. Penalización por tablas reales
        if not (has_dots or has_spaces or is_hierarchical or is_roman or is_list_roman):
            digits = sum(c.isdigit() for c in line)
            if digits >= 6:
                numeric_heavy_lines += 1

    total = len(lines)
    if total == 0:
        return 0.0

    ratio_end_number = lines_ending_in_number / total
    ratio_dotted = lines_with_dotted_leader / total
    ratio_space_leader = lines_with_space_leader / total
    ratio_numeric_heavy = numeric_heavy_lines / total
    ratio_hierarchical = hierarchical_numbered_lines / total
    ratio_roman = roman_lines / total
    ratio_list_roman = list_roman_lines / total

    score = 0.0

    # --- 3. Scoring ---
    if ratio_end_number > 0.35:
        score += 0.5
    elif ratio_end_number > 0.2:
        score += 0.3

    if ratio_dotted > 0.15:
        score += 0.4

    if ratio_space_leader > 0.15:
        score += 0.35
    elif ratio_space_leader > 0.08:
        score += 0.2

    if ratio_hierarchical > 0.25:
        score += 0.45
    elif ratio_hierarchical > 0.15:
        score += 0.25

    if ratio_list_roman > 0.25:
        score += 0.5
    elif ratio_list_roman > 0.15:
        score += 0.35

    elif ratio_roman > 0.15:
        score += 0.3

    if has_keyword:
        score += 0.25

    # --- 4. Penalización ---
    if ratio_numeric_heavy > 0.3:
        score -= 0.5
    elif ratio_numeric_heavy > 0.15:
        score -= 0.3

    return max(0.0, min(score, 1.0))

def distance_penalty(page_num, start_penalizing=10, strength=0.07, min_factor=0.6):
    """
    Penalización suave y limitada para páginas lejanas.
    """
    if page_num <= start_penalizing:
        return 1.0

    d = page_num - start_penalizing
    penalty = math.exp(-strength * d)

    return max(min_factor, penalty)


def detect_index_pages(pages, threshold=0.6):
    """
    Devuelve los números de página cuya puntuación de índice
    (index_score, atenuada por la distancia al inicio del documento)
    supera el umbral indicado.
    """
    detected = []

    for p in pages:
        base = index_score(p["text"])
        penalty = distance_penalty(p["page"])
        final_score = base * penalty

        if final_score >= threshold:
            detected.append(p["page"])

    return detected


def remove_index_pages(pages, pages_to_remove):
    """Filtra la lista de páginas eliminando las indicadas en pages_to_remove."""
    pages_to_remove_set = set(pages_to_remove)
    return [p for p in pages if p["page"] not in pages_to_remove_set]


In [14]:
import re

def detect_bib_pages(pages_data):
    """
    Detecta las páginas que contienen la sección de bibliografía/referencias.
    Localiza la página de inicio mediante el título de sección (BIBLIOGRAFÍA,
    REFERENCIAS, FUENTES...) y delimita el final de la sección al encontrar una
    cabecera de parada (ANEXOS, ÍNDICE, TABLAS...) o contenido de listas/citas.
    """
    bib_pages = []
    in_bib_section = False

    # 1. REGEX: TÍTULO DE BIBLIOGRAFÍA (Sin cambios)
    RE_BIB_TITLE_LINE = re.compile(
        r'^\s*'
        r'(?:(?:\d{1,2}|[IVX]+)[.\-)\s]+)?'
        r'(?:'
            r'(?:BIBLIOGRAF[ÍI]A|REFERENCIAS|FUENTES)'
            r'(?:\s+(?:Y|E|AND)\s+'
                r'(?:BIBLIOGRAF[ÍI]A|REFERENCIAS|FUENTES|REFERENCIAS\s+BIBLIOGR[ÁA]FICAS)'
            r')?'
            r'(?:\s+(?:CITADAS?|CONSULTADAS?|DOCUMENTALES|B[ÁA]SICA|COMPLEMENTARIA|GENERAL|RECOMENDADA|SELECCIONADA))?'
        r')'
        r'[\.:]?\s*$',
        re.IGNORECASE
    )

    # --------------------------------------------------
    # 2. REGEX: STOP HEADERS
    # --------------------------------------------------
    RE_STOP_TITLE_LINE = re.compile(
        r'^\s*'
        r'(?:(?:\d{1,2}|[IVX]+)[.\-)\s]+)?'
        r'(?:'
            r'ANEXOS?|APÉNDICES?|AGRADECIMIENTOS|GLOSARIO|ÍNDICE|INDICE|TABLAS?|'
            r'OBJETIVOS|INDICADORES|COLABORADORES|CONCLUSIONES|'
            r'EQUIPO(?:\s+DE\s+(?:TRABAJO|INVESTIGACI[ÓO]N|REDACCIÓN))?'
        r')'
        r'(?:[\.:\s].*)?$',
        re.IGNORECASE
    )

    # ... (Resto de REGEX igual: RE_LIST_NUMERIC, etc.) ...
    RE_LIST_NUMERIC = re.compile(r'(?:^|\n)\s*(?:\[\d+\]|\d+[.)]|\d+(?=\s+[A-ZÁÉÍÓÚÑ]))\s+')
    RE_LIST_BULLET = re.compile(r'(?:^|\n)\s*(?:[•\-–])\s+')
    # --------------------------------------------------
    # 3. REGEX: CONTENIDO (CORREGIDA)
    # --------------------------------------------------
    # CAMBIO: He añadido \b delante de pp., vol., ed., rev. para evitar falsos positivos
    # como 'spp.' (bacterias), 'revolución', 'med.' (médico), etc.
    RE_KEYWORDS = re.compile(
        r'(?:disponible en:|available at:|citado|cited|doi:|'
        r'\bvol\.|'     # \b para evitar coincidencias parciales
        r'\bpp\.|'      # \b CRUCIAL para evitar 'spp.'
        r'et al\.|isbn|issn|http|www\.|editor|'
        r'\bed\.|'      # \b para evitar coincidir con final de palabras
        r'springer|elsevier|'
        r'\brev\.|'     # \b para evitar otras palabras terminadas en rev
        r'bolet[íi]n)',
        re.IGNORECASE
    )
    RE_YEAR = re.compile(r'(?:^|\s|[(])(19|20)\d{2}(?:$|\s|[).,:;])')
    RE_SENTENCE = re.compile(r'[.!?]\s+')

    for i, page_entry in enumerate(pages_data):
        page_num = page_entry.get("page", i)
        raw_text = page_entry.get("text", "") or ""

        # A. LIMPIEZA
        lines = []
        for l in raw_text.splitlines():
            clean_l = l.replace('\u200b', '').replace('\xa0', ' ').replace('\ufeff', '').strip()
            if clean_l:
                lines.append(clean_l)
        page_text_clean = "\n".join(lines)

        # --- CORRECCIÓN CLAVE AQUÍ ---

        # 1. PRIMERO: Buscamos si la página TIENE un encabezado de bibliografía.
        # Esto tiene prioridad sobre cualquier "Stop Header" que pueda haber (como una Tabla).
        has_bib_header = any(RE_BIB_TITLE_LINE.match(line) for line in lines)

        if has_bib_header:
            bib_pages.append(page_num)
            in_bib_section = True
            continue # Encontramos el inicio, nos quedamos la página y pasamos a la siguiente.

        # 2. SEGUNDO: Si NO es inicio de biblio, miramos si es un STOP HEADER.
        # Si encontramos "ANEXOS", cortamos la sección.
        if any(RE_STOP_TITLE_LINE.match(line) for line in lines):
            in_bib_section = False
            continue

        # 3. TERCERO: Si no hay headers, seguimos la lógica de continuación
        if not in_bib_section:
            continue

        # ... (El resto del código de cálculo de densidad sigue igual) ...
        word_count = max(1, len(page_text_clean.split()))
        numeric_matches = list(RE_LIST_NUMERIC.finditer(page_text_clean))
        count_numeric = len(numeric_matches)
        count_bullets = len(RE_LIST_BULLET.findall(page_text_clean))
        count_keywords = len(RE_KEYWORDS.findall(page_text_clean))
        count_years = len(RE_YEAR.findall(page_text_clean))

        raw_score = (
            count_numeric * 3.5 +
            count_bullets * 0.5 +
            count_keywords * 2.0 +
            count_years * 1.0
        )
        density_score = (raw_score / word_count) * 100

        late_inline_refs = False
        if numeric_matches:
            first_ref_pos = numeric_matches[0].start()
            pre_ref_text = page_text_clean[:first_ref_pos]
            pre_words = len(pre_ref_text.split())
            pre_ratio = pre_words / word_count
            sentences = RE_SENTENCE.split(pre_ref_text)
            avg_sentence_len = (sum(len(s.split()) for s in sentences) / max(1, len(sentences)))

            late_inline_refs = (
                pre_ratio > 0.6 and
                avg_sentence_len > 18 and
                count_numeric <= 8 and
                not has_bib_header
            )

        if density_score > 1.5 and not late_inline_refs:
            bib_pages.append(page_num)
        else:
            in_bib_section = False

    return sorted(set(bib_pages))

In [15]:
import re

def clean_biblio_and_log(pages_data, bib_pages_indices):
    """
    Elimina el contenido de las páginas de bibliografía detectadas por
    detect_bib_pages, desde el título de la sección hasta el final de la
    página (o hasta una cabecera de parada), registrando en deleted_lines_log
    las líneas eliminadas para el log de auditoría del pipeline.
    """
    bib_set = set(bib_pages_indices)
    if not bib_set:
        return pages_data, {}

    sorted_bib_pages = sorted(list(bib_set))
    deleted_lines_log = {}
    cleaned_pages = []

    # --- REGEX ---
    RE_BIB_TITLE_LINE = re.compile(
        r'^.*?\b(?:BIBLIOGRAF[ÍI]A|REFERENCIAS|FUENTES)'
        r'(?:\s+(?:Y|E|AND)\s+(?:BIBLIOGRAF[ÍI]A|REFERENCIAS|FUENTES|REFERENCIAS\s+BIBLIOGR[ÁA]FICAS))?'
        r'(?:\s+(?:CITADAS?|CONSULTADAS?|DOCUMENTALES|B[ÁA]SICA|COMPLEMENTARIA|GENERAL))?[\.:]?\s*$',
        re.IGNORECASE
    )

    RE_STOP_TITLE_LINE = re.compile(
        r'^\s*(?:(?:\d{1,2}|[IVX]+)[.\-)\s]+)?(?:ANEXOS?|APÉNDICES?|AGRADECIMIENTOS|GLOSARIO|ÍNDICE|INDICE|TABLAS?|OBJETIVOS|INDICADORES|COLABORADORES|CONCLUSIONES)(?:[\.:\s].*)?$',
        re.IGNORECASE
    )

    RE_STOP_SECTION_NUMERIC = re.compile(r'^\s*\d+(?:\.\d+)+[.\-]?\s+[A-ZÁÉÍÓÚÑ]')

    # --- CAMBIO CRUCIAL AQUÍ ---
    # 1. Soporta guion opcional tras punto [.\-–]?
    # 2. El final es (?:\s+|$) -> Significa "espacio O fin de línea"
    # Esto permite detectar "1." aunque esté solo en la línea.
    RE_CITATION_START = re.compile(
        r'(?:^|\n)\s*(?:\[\d+\]|\d{1,4}[.)][\-\–]?|\d+(?=\s+[A-ZÁÉÍÓÚÑ])|[•\-–])(?:\s+|$)',
        re.IGNORECASE
    )

    RE_STRONG_KEYWORDS = re.compile(
        r'(?:disponible en:|available at:|citado|cited|doi:|'
        r'\bvol\.|\bpp\.|et al\.|isbn|issn|editor|\bed\.|'
        r'springer|elsevier|\brev\.|bolet[íi]n)',
        re.IGNORECASE
    )

    for i, page_entry in enumerate(pages_data):
        page_num = page_entry.get("page", i)
        raw_text = page_entry.get("text", "") or ""
        lines = raw_text.splitlines()

        if page_num not in bib_set:
            cleaned_pages.append(page_entry)
            continue

        # --- DETECCIÓN INICIO ---
        remove_start = 0
        remove_end = len(lines)
        found_start_marker = False

        is_first_bib_page = (page_num == sorted_bib_pages[0]) or (page_num - 1 not in bib_set)
        title_index = -1

        for idx, line in enumerate(lines):
            clean_l = line.strip()
            if len(clean_l) < 100 and RE_BIB_TITLE_LINE.match(clean_l):
                title_index = idx
                found_start_marker = True
                break

        if found_start_marker:
            remove_start = title_index
            if title_index >= (len(lines) - 5) and remove_start > 0:
                check_range = lines[max(0, title_index - 10) : title_index]
                if check_range:
                    citations_found = sum(1 for l in check_range if RE_CITATION_START.match(l))
                    if citations_found >= 2:
                        remove_start = 0

        elif is_first_bib_page:
            for idx, line in enumerate(lines):
                if RE_CITATION_START.match(line) and len(line) > 10:
                    remove_start = idx
                    found_start_marker = True
                    break
            if not found_start_marker:
                remove_start = 0

        # --- DETECCIÓN FIN ---
        last_safe_bib_index = remove_start
        suspicious_gap_threshold = 10
        lookahead_window = 15

        for idx in range(remove_start, len(lines)):
            line = lines[idx]
            clean_l = line.strip()

            if idx > remove_start:
                if RE_STOP_TITLE_LINE.match(clean_l) or RE_STOP_SECTION_NUMERIC.match(clean_l):
                    remove_end = idx
                    break

            # Match tolerante a fin de línea
            is_citation_start = bool(RE_CITATION_START.match(line))
            is_strong_keyword = bool(RE_STRONG_KEYWORDS.search(line))

            if is_citation_start or is_strong_keyword:
                last_safe_bib_index = idx
            else:
                gap_size = idx - last_safe_bib_index

                if gap_size > suspicious_gap_threshold:
                    lines_remaining_total = len(lines) - idx

                    if lines_remaining_total < 10:
                        continue

                    lookahead_range = min(lookahead_window, lines_remaining_total)
                    lookahead_slice = lines[idx : idx + lookahead_range]

                    has_future_citation = any(RE_CITATION_START.match(l) for l in lookahead_slice)

                    if not has_future_citation:
                        remove_end = last_safe_bib_index + 1
                        break
                    else:
                        pass

        # --- GENERAR SALIDA ---
        kept_lines = []
        removed_lines_in_page = []

        if remove_start > 0:
            kept_lines.extend(lines[:remove_start])

        removed_lines_in_page.extend(lines[remove_start:remove_end])

        if remove_end < len(lines):
            kept_lines.extend(lines[remove_end:])

        if removed_lines_in_page:
            deleted_lines_log[page_num] = removed_lines_in_page

        new_text = "\n".join(kept_lines)
        new_page_entry = page_entry.copy()
        new_page_entry["text"] = new_text
        cleaned_pages.append(new_page_entry)

    return cleaned_pages, deleted_lines_log

In [16]:
import re

def detect_author_pages(pages_data):
    """
    Detecta páginas que contienen la sección de autores/coordinadores/equipo
    de trabajo, mediante palabras clave de inicio de sección (AUTORES,
    COORDINACIÓN, COMITÉ...) y cabeceras de parada (ÍNDICE, RESUMEN,
    BIBLIOGRAFÍA...) que marcan el final de la sección.
    """
    author_pages = []
    in_author_section = False

    # 1. REGEX: PALABRAS CLAVE BASE
    KEYWORDS_CORE = (
        r'AUTORES|AUTORÍA|GRUPO\s+DE\s+TRABAJO|COORDINACI[ÓO]N|COORDINADORES|'
        r'COLABORADORES|EQUIPO\s+DE\s+(?:TRABAJO|REDACCI[ÓO]N|INVESTIGACI[ÓO]N)|'
        r'COMIT[ÉE](?:\s+CIENT[ÍI]FICO|\s+ASESOR)?|REVISORES'
    )

    ENUM_PREFIX = r'(?:(?:\d{1,2}|[IVX]+)[.\-)\s]+|[•\-–]\s*)'

    # 2. REGEX: INICIO
    # Caso A: Título solo (flexible con sufijos)
    RE_AUTH_TITLE_SOLO = re.compile(
        rf'^\s*{ENUM_PREFIX}?'
        rf'(?:{KEYWORDS_CORE})'
        r'(?:\s+[A-ZÁÉÍÓÚÑ\w]+){0,2}'       # Permite "COORDINACIÓN TÉCNICA", etc.
        r'[\.:]?\s*$',
        re.IGNORECASE
    )

    # Caso B: Inline con enumeración obligatoria
    RE_AUTH_TITLE_INLINE = re.compile(
        rf'^\s*({ENUM_PREFIX})'
        rf'(?:{KEYWORDS_CORE})\b'
        r'.+$',
        re.IGNORECASE
    )

    # 3. REGEX: STOP HEADERS (AQUÍ ESTÁ LA CORRECCIÓN CLAVE)
    # Hemos añadido: ACRÓNIMOS, SIGLAS, ABREVIATURAS para frenar el borrado.
    RE_HARD_STOP = re.compile(
        r'^\s*(?:'
        r'INDICE|ÍNDICE|TABLA\s+DE\s+CONTENIDOS|'
        r'RESUMEN|ABSTRACT|GLOSARIO|'
        r'BIBLIOGRAF[ÍI]A|REFERENCIAS|'
        r'INTRODUCCI[ÓO]N|'
        r'ACR[ÓO]NIMOS|SIGLAS|ABREVIATURAS|LISTADO\s+DE\s+ABREVIATURAS' # <--- NUEVOS FRENOS
        r')',
        re.IGNORECASE
    )

    RE_NEW_NUMBERED_SECTION = re.compile(
        r'^\s*(?:\d{1,2}|[IVX]+)[.\-)\s]+'
        r'(?!' + KEYWORDS_CORE + r')'
        r'[A-ZÁÉÍÓÚÑ]{3,}'
        r'.*$',
        re.IGNORECASE
    )

    # 4. DENSIDAD
    RE_NAME_INDICATORS = re.compile(
        r'\b(?:'
        r'Dr\.|Dra\.|Lcd[oa]\.|Prof\.|PhD|MSc|'
        r'Hospital|Universidad|Facultad|Cl[íi]nica|'
        r'Unidad|Servicio|Departamento|Dpto\.|'
        r'Centro\s+de\s+Salud|C\.S\.|E\.U\.|'
        r'Jefe|Médico|Enfermer[oa]|Técnico|'
        r'Consejería|Subdirección|Dirección'
        r')\b',
        re.IGNORECASE
    )

    RE_POSSIBLE_NAME_LINE = re.compile(r'^\s*[A-ZÁÉÍÓÚÑ][a-záéíóúñ]+(?:\s+[A-ZÁÉÍÓÚÑ][a-záéíóúñ]+){1,4}[,.]?\s*$')

    for i, page_entry in enumerate(pages_data):
        page_num = page_entry.get("page", i)
        raw_text = page_entry.get("text", "") or ""

        lines = []
        for l in raw_text.splitlines():
            clean_l = l.replace('\u200b', '').replace('\xa0', ' ').strip()
            if clean_l:
                lines.append(clean_l)

        page_text_clean = "\n".join(lines)
        word_count = max(1, len(page_text_clean.split()))

        is_start_page = False

        # --- DETECCIÓN DE INICIO ---
        for line in lines[:10]:
            if RE_AUTH_TITLE_SOLO.match(line):
                is_start_page = True
                break

            match_inline = RE_AUTH_TITLE_INLINE.match(line)
            if match_inline and match_inline.group(1).strip():
                is_start_page = True
                break

        if is_start_page:
            author_pages.append(page_num)
            in_author_section = True
            continue

        # --- LÓGICA DE CONTINUACIÓN ---
        if in_author_section:
            # 1. CHEQUEO DE PARADA (STOP) - PRIORITARIO
            # Si encontramos ACRÓNIMOS, ÍNDICE, INTRODUCCIÓN, cortamos inmediatamente.
            # Verificamos las primeras 5 líneas de la página para encontrar el título.
            if any(RE_HARD_STOP.match(line) for line in lines[:8]): # Aumentado a 8 por si el título baja
                in_author_section = False
                continue

            if any(RE_NEW_NUMBERED_SECTION.match(line) for line in lines):
                in_author_section = False
                continue

            # 2. CHEQUEO DE DENSIDAD
            count_indicators = len(RE_NAME_INDICATORS.findall(page_text_clean))
            count_name_lines = sum(1 for line in lines if RE_POSSIBLE_NAME_LINE.match(line))

            total_score = (count_indicators * 2.0) + (count_name_lines * 1.5)
            density = (total_score / word_count) * 100

            if density > 4.0 or (word_count < 100 and total_score >= 3):
                author_pages.append(page_num)
            else:
                in_author_section = False

    return sorted(set(author_pages))

In [17]:
import re

def clean_authors_and_log(pages_data, author_pages_indices):
    """
    Elimina el contenido de las páginas de autores detectadas.
    Sincronizada con la lógica de 'detect_author_pages' para asegurar
    que se borra exactamente lo que se detectó.
    """
    author_set = set(author_pages_indices)
    if not author_set:
        return pages_data, {}

    sorted_auth_pages = sorted(list(author_set))
    deleted_lines_log = {}
    cleaned_pages = []

    # ---------------------------------------------------------
    # 1. REGEX DE INICIO (Debe coincidir con la detección)
    # ---------------------------------------------------------
    KEYWORDS_CORE = (
        r'AUTORES|AUTORÍA|GRUPO\s+DE\s+TRABAJO|COORDINACI[ÓO]N|COORDINADORES|'
        r'COLABORADORES|EQUIPO\s+DE\s+(?:TRABAJO|REDACCI[ÓO]N|INVESTIGACI[ÓO]N)|'
        r'COMIT[ÉE](?:\s+CIENT[ÍI]FICO|\s+ASESOR)?|REVISORES'
    )

    ENUM_PREFIX = r'(?:(?:\d{1,2}|[IVX]+)[.\-)\s]+|[•\-–]\s*)'

    # Busca la línea de título. Es ligeramente más permisiva al final de la línea
    # para asegurar que atrapamos "Coordinación Técnica" o "Autores: Juan..."
    RE_AUTHOR_START_LINE = re.compile(
        rf'^\s*{ENUM_PREFIX}?'              # 1. o bullet opcional
        rf'(?:{KEYWORDS_CORE})'             # Palabra clave
        r'(?:[:\.\s].*)?$',                 # Cualquier cosa que siga (dos puntos, texto, etc)
        re.IGNORECASE
    )

    # ---------------------------------------------------------
    # 2. REGEX DE PARADA (STOP) - ¡CRÍTICO!
    # ---------------------------------------------------------
    # Aquí añadimos ACRÓNIMOS y el resto de secciones que definen el fin.
    RE_STOP_SECTION_KEYWORDS = re.compile(
        r'^\s*'
        r'(?:'
            r'INDICE|ÍNDICE|TABLA\s+DE\s+CONTENIDOS|'
            r'RESUMEN|ABSTRACT|GLOSARIO|'
            r'BIBLIOGRAF[ÍI]A|REFERENCIAS|'
            r'INTRODUCCI[ÓO]N|'
            r'ACR[ÓO]NIMOS|SIGLAS|ABREVIATURAS|LISTADO\s+DE\s+ABREVIATURAS|'
            r'JUSTIFICACI[ÓO]N|OBJETIVOS'
        r')'
        r'(?:[\.:\s].*)?$', # Permite texto tras el título (ej: "Introducción general")
        re.IGNORECASE
    )

    # Detecta "2. Introducción" (Numeración + Título NO Autores)
    RE_STOP_NUMBERED_SECTION = re.compile(
        r'^\s*(?:\d{1,2}|[IVX]+)[.\-)\s]+'
        r'(?!' + KEYWORDS_CORE + r')'       # Negative lookahead (que no sea autores)
        r'[A-ZÁÉÍÓÚÑ]{3,}'                  # Título en mayúsculas
        r'.*$',
        re.IGNORECASE
    )

    for i, page_entry in enumerate(pages_data):
        page_num = page_entry.get("page", i)
        raw_text = page_entry.get("text", "") or ""
        lines = raw_text.splitlines()

        # Si la página no fue marcada para borrar, la guardamos tal cual
        if page_num not in author_set:
            cleaned_pages.append(page_entry)
            continue

        # --- LÓGICA DE CORTE DENTRO DE LA PÁGINA ---

        # Por defecto, si es una página de autores intermedia, borramos desde la línea 0
        remove_start = 0
        remove_end = len(lines)

        # CASO: Primera página detectada (o nueva sección tras un salto)
        # Buscamos dónde EMPIEZA el título de autores.
        # (Ej: La página tiene título del documento arriba y luego "Coordinación")
        is_start_of_section = (page_num == sorted_auth_pages[0]) or (page_num - 1 not in author_set)

        if is_start_of_section:
            found_start = False
            for idx, line in enumerate(lines):
                if RE_AUTHOR_START_LINE.match(line):
                    remove_start = idx
                    found_start = True
                    break
            # Si es la primera página pero no encontramos el título (raro, pero posible si el título
            # estaba al final de la anterior no detectada), asumimos borrar todo o ser conservadores.
            # Aquí asumimos borrar desde el principio si falló el regex, o puedes poner remove_start=0

        # CASO: Buscar dónde TERMINA (Stop Headers)
        # Buscamos desde remove_start hacia abajo
        for idx in range(remove_start + 1, len(lines)):
            line = lines[idx]
            # Si encontramos "Acrónimos" o "2. Introducción", paramos de borrar ahí.
            if RE_STOP_SECTION_KEYWORDS.match(line) or RE_STOP_NUMBERED_SECTION.match(line):
                remove_end = idx
                break

        # --- RECONSTRUCCIÓN DEL TEXTO ---
        kept_lines = []
        removed_lines_in_page = []

        # 1. Guardar líneas ANTES del inicio de autores (si las hay)
        if remove_start > 0:
            kept_lines.extend(lines[:remove_start])

        # 2. Registrar líneas BORRADAS
        removed_lines_in_page.extend(lines[remove_start:remove_end])

        # 3. Guardar líneas DESPUÉS del corte (ej: empieza la Intro en la misma hoja)
        if remove_end < len(lines):
            kept_lines.extend(lines[remove_end:])

        # Guardar log
        if removed_lines_in_page:
            deleted_lines_log[page_num] = removed_lines_in_page

        # Crear nueva entrada de página
        new_text = "\n".join(kept_lines)
        new_page_entry = page_entry.copy()
        new_page_entry["text"] = new_text

        # Opcional: Si la página queda vacía, ¿la guardamos vacía o la quitamos?
        # Normalmente mejor guardarla vacía para no romper índices de páginas si usas eso luego.
        cleaned_pages.append(new_page_entry)

    return cleaned_pages, deleted_lines_log

In [18]:
import re

def detect_reference_pages(pages_data):
    """
    Detecta páginas que contienen referencias o notas al pie basándose en patrones
    específicos de estructura y contexto (densidad de inglés, enlaces, etc.).
    """
    ref_pages = []

    # --------------------------------------------------
    # 1. REGEX BASICAS (Heredadas y Adaptadas)
    # --------------------------------------------------
    # Palabras clave técnicas de referencias (Heredado de tu función)
    RE_KEYWORDS = re.compile(
        r'(?:disponible en:|available at:|citado|cited|doi:|'
        r'\bvol\.|'
        r'\bpp\.|'
        r'et al\.|isbn|issn|http|www\.|editor|'
        r'\bed\.|'
        r'springer|elsevier|wiley|lancet|pubmed|'
        r'\brev\.|'
        # CORRECCIÓN AQUÍ: Agregados \b para evitar falsos positivos en español
        r'\bmedrxiv\b|\bpreprint\b|\bmmwr\b|\bwkly\b|rep\.|'
        r'bolet[íi]n)',
        re.IGNORECASE
    )

    # Palabras comunes en inglés (para detectar "alta densidad de inglés")
    # Se añaden a las keywords técnicas para reforzar la detección.
    RE_ENGLISH_COMMON = re.compile(
        r'\b(the|of|and|in|to|for|with|on|at|by|from|world|health|influenza|vaccination)\b',
        re.IGNORECASE
    )

    # Detecta enlaces explícitos o indicaciones de disponibilidad
    RE_LINKS = re.compile(
        r'(?:https?://|www\.|doi:|disponible\s+en:|available\s+at:)',
        re.IGNORECASE
    )

    # PATRON 1: Dígito seguido de espacio seguido de mayúscula al inicio de línea
    # Ej: "1 Referencia..."
    RE_DIGIT_SPACE_UPPER = re.compile(r'^\d{1,2}\s*[A-ZÁÉÍÓÚÑ]')

    # PATRON 2: Línea con solo dígitos (para el caso del dígito solo + línea anterior con punto)
    RE_LONE_DIGIT = re.compile(r'^\d{1,2}$')

    # Patrón para años entre paréntesis o sueltos (muy común en refs: (2015), 2023.)
    RE_YEAR_PATTERN = re.compile(r'\b(19|20)\d{2}\b')

    # --------------------------------------------------
    # 2. FUNCIONES AUXILIARES
    # --------------------------------------------------
    def check_window_context(lines_subset):
        """
        Analiza un subconjunto de líneas (aprox 15) para ver si cumplen los requisitos:
        - Algún enlace o 'Disponible en'
        - Alta densidad de palabras en inglés/técnicas
        - Alta densidad de números (indicativo de años, páginas, vols)
        """
        if not lines_subset:
            return False

        text_block = " ".join(lines_subset)
        word_count = len(text_block.split())
        if word_count == 0:
            return False

        # Conteo de elementos
        num_links = len(RE_LINKS.findall(text_block))
        num_keywords = len(RE_KEYWORDS.findall(text_block))
        num_english = len(RE_ENGLISH_COMMON.findall(text_block))
        num_years = len(RE_YEAR_PATTERN.findall(text_block))

        # Heurística: Si hay enlace, es muy probable que sea referencia.
        if num_links > 0:
            return True

        # Heurística: Densidad de términos técnicos/inglés
        # Si más del 10-15% de las palabras son keywords o inglés común, es positivo.
        english_score = (num_keywords + num_english + num_years)
        if english_score > 4:
            return True

        return False

    def primera_prueba(lines_subset):
        """
        Variante más estricta de check_window_context: exige a la vez presencia
        de elementos de referencia (enlaces, keywords, años) Y una alta densidad
        de palabras con mayúscula inicial, típica de listas de autores/títulos
        en bibliografía.
        """
        if not lines_subset:
            return False

        text_block = " ".join(lines_subset)
        words = text_block.split()
        word_count = len(words)
        if word_count == 0:
            return False

        # 1. Calculamos los matches (Enlaces, Keywords, Años)
        num_links = len(RE_LINKS.findall(text_block))
        num_keywords = len(RE_KEYWORDS.findall(text_block))
        num_years = len(RE_YEAR_PATTERN.findall(text_block))

        matches = num_links + num_keywords + num_years

        # 2. Calculamos la densidad de mayúsculas
        # Contamos palabras que empiezan por mayúscula (Autores, Iniciales, Títulos)
        # Regex \b[A-ZÁÉÍÓÚÑ] busca inicio de palabra mayúscula
        num_caps = len(re.findall(r'\b[A-ZÁÉÍÓÚÑ]', text_block))

        # Definimos el umbral: > 25% (0.25) es típico de listas de bibliografía
        has_high_caps_density = (num_caps / word_count) > 0.25

        # 3. CONDICIÓN DOBLE (AND): Matches > 0 Y Densidad alta
        if matches > 0 and has_high_caps_density:
            return True

        return False


    def is_high_density_line(line):
        """
        Verifica si una sola línea tiene alta densidad de inglés/keywords.
        (Patrón 2 del documento: Línea con alta densidad...)
        """
        words = line.split()
        if len(words) < 3: return False # Ignorar líneas muy cortas

        matches = len(RE_KEYWORDS.findall(line)) + len(RE_ENGLISH_COMMON.findall(line))
        # Si la mitad de la línea son términos técnicos/inglés, o tiene muchos
        return matches >= 6


    # --------------------------------------------------
    # 3. PROCESAMIENTO DE PÁGINAS
    # --------------------------------------------------
    for i, page_entry in enumerate(pages_data):
        page_num = page_entry.get("page", i)
        raw_text = page_entry.get("text", "") or ""

        # --- NUEVO: Limpieza previa ---
        # Eliminamos referencias intratexto (ej: texto12, texto(1)) para limpiar ruido.
        # No afecta a inicios de línea ni a años completos.
        ref_pattern = r'(?<=[a-z"\'”’\)])(\d{1,3}(?:,\d{1,3})*)(?=\s|[.,;]|$)'
        raw_text = re.sub(ref_pattern, '', raw_text)

        # A. Limpieza (Igual que tu función original)
        lines = []
        for l in raw_text.splitlines():
            clean_l = l.replace('\u200b', '').replace('\xa0', ' ').replace('\ufeff', '').strip()
            if clean_l:
                lines.append(clean_l)

        has_references = False

        # Iteramos sobre las líneas para buscar los patrones
        for idx, line in enumerate(lines):

            # --- PATRÓN A: Dígito + Espacio + Mayúscula ---
            # Requisito: Siguientes 15 líneas tienen enlaces o inglés.
            if RE_DIGIT_SPACE_UPPER.match(line):
                next_lines = lines[idx: idx+2]
                if primera_prueba(next_lines):
                  has_references = True
                  break # Ya encontramos referencias en esta página, pasamos a la siguiente

                # Tomamos las siguientes 15 líneas (slice seguro en Python)
                else:
                  next_lines = lines[idx : idx+21]
                  if check_window_context(next_lines):
                      has_references = True
                      break # Ya encontramos referencias en esta página, pasamos a la siguiente

            # --- PATRÓN B: Línea es un dígito solo ---
            # Requisito: La línea ANTERIOR debe terminar en punto '.'
            # Y las siguientes 15 líneas tienen enlaces o inglés.
            if RE_LONE_DIGIT.match(line):
                if idx > 0 and idx + 1 < len(lines) and len(lines[idx+1]) > 4 and lines[idx+1][0].isupper() and (lines[idx-1].endswith(('.', ':',',')) or (lines[idx-1][-1].islower() and len(lines[idx-1].split()) > 7)):
                    next_lines = lines[idx+1: idx+3]
                    if primera_prueba(next_lines):
                      has_references = True
                      break # Ya encontramos referencias en esta página, pasamos a la siguiente

                    # Tomamos las siguientes 15 líneas (slice seguro en Python)
                    else:
                      next_lines = lines[idx+1: idx+21]
                      if check_window_context(next_lines):
                          has_references = True
                          break # Ya encontramos referencias en esta página, pasamos a la siguiente

            # --- PATRÓN C: Línea única con alta densidad de inglés ---
            # Para casos donde la referencia está toda en una línea
            if is_high_density_line(line):
                has_references = True
                break

        if has_references:
            ref_pages.append(page_num)

    return sorted(set(ref_pages))

In [19]:
import re

def remove_specific_references_strict(pages_data, ref_pages_indices):
    """
    Elimina líneas de referencias.
    INICIO: Lógica estricta de detección (Patrones A, B, C).
    FIN: Lógica binaria. Si hay CUALQUIER elemento de referencia en la línea
         o en las inmediatas posteriores, se borra.
    """
    cleaned_pages = []
    deleted_lines_log = {}
    target_pages = set(ref_pages_indices)

    # --------------------------------------------------
    # 1. REGEX (Tus patrones originales)
    # --------------------------------------------------
    RE_KEYWORDS = re.compile(
        r'(?:disponible en:|available at:|citado|cited|doi:|'
        r'\bvol\.|'
        r'\bpp\.|'
        r'et al\.|isbn|issn|http|www\.|editor|'
        r'\bed\.|'
        r'springer|elsevier|wiley|lancet|pubmed|'
        r'\brev\.|'
        r'\bmedrxiv\b|\bpreprint\b|\bmmwr\b|\bwkly\b|rep\.|'
        r'bolet[íi]n)',
        re.IGNORECASE
    )

    RE_ENGLISH_COMMON = re.compile(
        r'\b(the|of|and|in|to|for|with|on|at|by|from|world|health|influenza|vaccination)\b',
        re.IGNORECASE
    )

    RE_LINKS = re.compile(
        r'(?:https?://|www\.|doi:|disponible\s+en:|available\s+at:)',
        re.IGNORECASE
    )

    RE_DIGIT_SPACE_UPPER = re.compile(r'^\d{1,2}\s*[A-ZÁÉÍÓÚÑ]')
    RE_LONE_DIGIT = re.compile(r'^\d{1,2}$')
    RE_YEAR_PATTERN = re.compile(r'\b(19|20)\d{2}\b')

    # --------------------------------------------------
    # 2. FUNCIONES AUXILIARES DE DETECCIÓN (INICIO)
    # --------------------------------------------------
    def check_window_context(lines_subset):
        """
        Réplica de la función homónima de detect_reference_pages: evalúa si un
        bloque de líneas tiene indicios de referencia (enlaces, keywords, inglés,
        años). Se duplica aquí para que la fase de BORRADO use exactamente el
        mismo criterio que la fase de DETECCIÓN.
        """
        # ... (Tu lógica original intacta para detectar inicio)
        if not lines_subset: return False
        text_block = " ".join(lines_subset)
        if not text_block.strip(): return False

        num_links = len(RE_LINKS.findall(text_block))
        num_keywords = len(RE_KEYWORDS.findall(text_block))
        num_english = len(RE_ENGLISH_COMMON.findall(text_block))
        num_years = len(RE_YEAR_PATTERN.findall(text_block))

        if num_links > 0: return True
        english_score = (num_keywords + num_english + num_years)
        return english_score > 4

    def primera_prueba(lines_subset):
        """
        Réplica de la función homónima de detect_reference_pages (ver arriba):
        combina presencia de elementos de referencia con alta densidad de
        mayúsculas iniciales.
        """
        # ... (Tu lógica original intacta para detectar inicio)
        if not lines_subset: return False
        text_block = " ".join(lines_subset)
        words = text_block.split()
        if not words: return False

        matches = (len(RE_LINKS.findall(text_block)) +
                   len(RE_KEYWORDS.findall(text_block)) +
                   len(RE_YEAR_PATTERN.findall(text_block)))

        num_caps = len(re.findall(r'\b[A-ZÁÉÍÓÚÑ]', text_block))
        has_high_caps_density = (num_caps / len(words)) > 0.25

        return matches > 0 and has_high_caps_density

    def is_high_density_line(line):
        """
        Réplica de la función homónima de detect_reference_pages: indica si una
        única línea tiene una densidad muy alta de keywords/inglés técnico.
        """
        matches = len(RE_KEYWORDS.findall(line)) + len(RE_ENGLISH_COMMON.findall(line))
        return matches >= 6

    # --------------------------------------------------
    # 3. NUEVA LÓGICA DE VALIDACIÓN (BOOLEANA)
    # --------------------------------------------------
    def has_any_ref_element(line):
        """
        Devuelve True si la línea contiene CUALQUIER rastro de referencia.
        Sin scores, simple presencia.
        """
        # 1. Enlaces o DOIs
        if RE_LINKS.search(line): return True
        # 2. Palabras clave técnicas (vol., pp., et al.)
        if RE_KEYWORDS.search(line): return True
        # 3. Años (1999, 2023)
        if RE_YEAR_PATTERN.search(line): return True

        # 4. Palabras comunes en inglés.
        # Como ya sabemos que estamos dentro de la zona de referencias,
        # encontrar un "the", "of", "and" es señal suficiente para seguir borrando.
        if RE_ENGLISH_COMMON.search(line): return True

        return False

    # --------------------------------------------------
    # 4. PROCESAMIENTO
    # --------------------------------------------------
    for i, page_entry in enumerate(pages_data):
        page_num = page_entry.get("page", i)

        if page_num not in target_pages:
            cleaned_pages.append(page_entry)
            continue

        raw_text = page_entry.get("text", "") or ""

        # Limpieza previa de referencias intratexto (igual que antes)
        ref_pattern_clean = r'(?<=[a-z"\'”’\)])(\d{1,3}(?:,\d{1,3})*)(?=\s|[.,;]|$)'
        raw_text = re.sub(ref_pattern_clean, '', raw_text)
        raw_text = re.sub(r"(https?://\S+)\s+(\S+)", r"\1\2", raw_text)

        lines = []
        for l in raw_text.splitlines():
            clean_l = l.replace('\u200b', '').replace('\xa0', ' ').replace('\ufeff', '').strip()
            if clean_l:
                lines.append(clean_l)

        start_index = -1

        # --- FASE 1: ENCONTRAR INICIO (Lógica Idéntica a Detección) ---
        for idx, line in enumerate(lines):
            found_here = False

            # Patrón A
            if RE_DIGIT_SPACE_UPPER.match(line):
                if primera_prueba(lines[idx: idx+2]): found_here = True
                elif check_window_context(lines[idx : idx+21]): found_here = True

            # Patrón B (Dígito solo)
            elif RE_LONE_DIGIT.match(line):
                valid_prev = False
                if idx > 0:
                     prev = lines[idx-1]
                     if prev.endswith(('.', ':',',')) or (prev[-1].islower() and len(prev.split()) > 7):
                         valid_prev = True

                valid_next = False
                if idx + 1 < len(lines) and len(lines[idx+1]) > 4 and lines[idx+1][0].isupper():
                    valid_next = True

                if valid_prev and valid_next:
                    if primera_prueba(lines[idx+1: idx+3]): found_here = True
                    elif check_window_context(lines[idx+1: idx+21]): found_here = True

            # Patrón C
            elif is_high_density_line(line):
                found_here = True

            if found_here:
                start_index = idx
                break

        # --- FASE 2: ENCONTRAR FIN (Lógica Booleana con Lookahead) ---

        if start_index == -1:
            # No se encontró el patrón de inicio, no borramos nada
            cleaned_pages.append(page_entry)
            continue

        remove_end = len(lines) # Asumimos borrar hasta el final por defecto
        consecutive_clean_lines = 0
        STOP_THRESHOLD = 17 # Si vemos 4 líneas seguidas sin NADA de referencias, paramos.
        found_valid_text_block = False

        # Recorremos línea a línea DESDE el inicio detectado
        for j in range(start_index, len(lines)):

            # -------------------------------------------------------------
            # NUEVA LÓGICA: BLOQUE DE TEXTO REAL (Estricto por línea)
            # -------------------------------------------------------------
            text_block_window = lines[j : j + 4]

            # Solo evaluamos si tenemos el bloque completo de 4 líneas
            if len(text_block_window) == 4:
                all_lines_are_text = True

                for tb_line in text_block_window:
                    # 1. Chequeo de referencias: Si tiene algo, NO es texto válido
                    if has_any_ref_element(tb_line):
                        all_lines_are_text = False
                        break

                    # 2. Chequeo de longitud INDIVIDUAL: Debe tener > 12 palabras
                    if len(tb_line.split()) <= 12:
                        all_lines_are_text = False
                        break

                # Si las 4 líneas pasaron ambas pruebas:
                if all_lines_are_text:
                    found_valid_text_block = True
                    break # Salimos del bucle, remove_end se queda donde estaba

            # Miramos la línea actual Y las 2 siguientes (Lookahead)
            # Si cualquiera de ellas tiene un elemento de referencia, es True.
            lines_window = lines[j : j + 3]

            is_ref_context = False
            for w_line in lines_window:
                if has_any_ref_element(w_line):
                    is_ref_context = True
                    break

            if is_ref_context:
                # Si hay rastro de referencia aquí o justo después, seguimos borrando.
                # Actualizamos el punto de corte para incluir esta línea (j+1 porque el slice excluye el final)
                remove_end = j + 1
                consecutive_clean_lines = 0
            else:
                # No hay rastro de referencia ni aquí ni en las siguientes inmediatas.
                consecutive_clean_lines += 1

            # Si hemos acumulado suficientes líneas limpias, confirmamos que la biblio terminó atrás
            if consecutive_clean_lines >= STOP_THRESHOLD:
                # remove_end ya se quedó fijo en la última línea que SÍ tenía referencias
                break
        # Si terminamos el bucle y quedaron líneas "vivas" al final,
        # verificamos si son suficientes para ser consideradas texto real.
        if remove_end < len(lines):
            remaining_lines = len(lines) - remove_end

            # Si lo que sobra es menos que nuestro umbral de seguridad,
            # asumimos que no era texto real, sino restos de bibliografía.
            # Si NO encontramos el bloque de texto válido, aplicamos el borrado por umbral
            if not found_valid_text_block and remaining_lines < STOP_THRESHOLD:
                remove_end = len(lines)

        # --- CONSTRUIR PÁGINA LIMPIA ---
        kept_lines = lines[:start_index]
        removed_lines = lines[start_index:remove_end]

        # Si no llegamos al final físico de la página, añadimos el texto restante
        if remove_end < len(lines):
            kept_lines.extend(lines[remove_end:])

        deleted_lines_log[page_num] = removed_lines

        new_text = "\n".join(kept_lines)
        new_page_entry = page_entry.copy()
        new_page_entry["text"] = new_text
        cleaned_pages.append(new_page_entry)

    return cleaned_pages, deleted_lines_log

In [20]:
def remove_front_matter(pages, pages_to_remove=1):
    """
    Elimina las primeras páginas del documento.

    Justificación (en protocolos clínicos):
    - La primera página suele ser portada institucional
    - No contiene información clínica relevante
    - Introduce ruido en embeddings
    """
    return pages[pages_to_remove:]

In [21]:
import os

# Definimos la lista exacta de los 20 documentos del Gold Standard
target_files = [
    "16304_Protocolo para la administración estacional de vacuna antigripal en los centros educativos.pdf",
    "16264_Vacunación estacional frente a infecciones respiratoria (gripe y COVID-19). Temporada 2024-2025.pdf",
    "10584_Vacunación estacional frente a infecciones respiratorias (gripe y COVID-19) en personas a partir de 60 años y grupos de riesgo de cualquier edad, así como antigripal en personas de 6 a 59 meses de edad. Temporada 2023-2024.pdf",
    "13564_Logística vacunal. Cadena de frío.pdf",
    "7447_Protocolo de vacunación antigripal y antineumocócica. Temporada 2019-2020.pdf",
    "19184_Protocolo para la captación de varones nacidos entre 1999 y 2010 no vacunados frente al Virus del Papiloma Humano (VPH).pdf",
    "4521_Protocolo para la detección y atención de la violencia de género en atención primaria.pdf",
    "4084_Protocolo regional de prevención y detección de violencia en la mujer mayor de 65 años.pdf",
    "11864_Procedimiento para el cumplimiento de la objeción de conciencia de los profesionales sanitarios implicados en la prestación de ayuda a morir.pdf",
    "4532_Atención al tabaquismo en atención primaria. Guía de práctica clínica.pdf",
    "16404_Protocolo de prevención y control ante brotes de gastroenteritis aguda en establecimientos hoteleros.pdf",
    "15264_Protocolo Regional para la indicación, uso y autorización de dispensación de medicamentos sujetos a prescripción médica por parte de las-os enfermeras-os- heridas.pdf",
    "15265_Protocolo Regional para la indicación, uso y autorización de dispensación de medicamentos sujetos a prescripción médica por parte de las-os enfermeras-os- ostomías.pdf",
    "10605_Protocolo de uso Fuera de Ficha Técnica de ONDANSETRÓN para el Tratamiento de Vómitos de Etiología Infecciosa en Población Pediátrica en Atención Primaria (Equipos de Atención Primaria y Servicios de Urgencias de Atención Primaria).pdf",
    "10604_Protocolo de uso de Nivolumab en condiciones distintas a las de la ficha técnica (dosificación por kg de peso) en las indicaciones aprobadas.pdf",
    "10624_Protocolo de uso de Pembrolizumab en condiciones distintas a las de la ficha técnica (dosificación por kg de peso) en las indicaciones aprobadas.pdf",
    "16384_Protocolo para la detección precoz y manejo de casos de mpox en la Región de Murcia.pdf",
    "11064_Protocolo de vigilancia, prevención y control de microorganismos multirresistentes y de especial relevancia epidemiológica en entornos hospitalarios de la Región de Murcia.pdf",
    "13584_Protocolo de vigilancia de la enfermedad de Lyme en la Región de Murcia.pdf",
    "1181_Guía hospitalaria de terapéutica antibiótica en adultos.pdf"
]

In [22]:
import os
import json

# Directorio base de salida
BASE_OUTPUT_DIR = "/content/drive/MyDrive/TFG_Resultados_Pipeline"

def save_json(data, folder_path, filename):
    """Guarda un JSON en la subcarpeta especificada."""
    os.makedirs(folder_path, exist_ok=True)
    full_path = os.path.join(folder_path, filename)

    with open(full_path, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=4)
    print(f"--> JSON guardado: {full_path}")

def save_detailed_log(global_log, folder_path, filename, title):
    """Guarda el log detallado en la subcarpeta especificada."""
    os.makedirs(folder_path, exist_ok=True)
    full_path = os.path.join(folder_path, filename)

    with open(full_path, "w", encoding="utf-8") as f:
        f.write(f"=== {title} ===\n")
        f.write(f"Total de archivos con cortes: {len(global_log)}\n\n")

        for filename_key, pages_map in global_log.items():
            f.write(f"ARCHIVO: {filename_key}\n")
            f.write("="*len(f"ARCHIVO: {filename_key}") + "\n")

            for page_num in sorted(pages_map.keys()):
                lines_removed = pages_map[page_num]
                f.write(f"  [Página {page_num}] - {len(lines_removed)} líneas eliminadas:\n")
                f.write("  " + "-"*40 + "\n")

                for line in lines_removed:
                    f.write(f"    | {line.strip()}\n")

                f.write("\n")

            f.write("\n" + "#"*50 + "\n\n")

    print(f"--> Log detallado guardado: {full_path}")

def main_pipeline(extract_path, target_files):
    """
    Orquesta el pipeline completo de extracción y limpieza de texto en 8
    pasos secuenciales (escaneo inicial, headers/footers, números de página,
    índices, bibliografía, autores, referencias específicas y limpieza
    final), guardando en BASE_OUTPUT_DIR el resultado intermedio de cada
    paso junto con sus logs de auditoría.
    """
    print("=== INICIANDO PIPELINE DE PREPROCESAMIENTO CLÍNICO (ESTRUCTURADO) ===\n")

    # ---------------------------------------------------------
    # PASO 1: ESCANEO INICIAL
    # ---------------------------------------------------------
    step_name = "1_initial_scan"
    step_dir = os.path.join(BASE_OUTPUT_DIR, step_name)
    print(f"[1/8] Escaneando PDFs... (Salida: {step_name})")

    step1_data = []

    for root, _, files in os.walk(extract_path):
        for file in files:
            if file in target_files:
                pdf_path = os.path.join(root, file)
                result = process_pdf(pdf_path)

                pages = result["pages"] if result["pages"] else []
                step1_data.append({
                    "file": file,
                    "status": result["status"],
                    "num_pages": len(pages),
                    "pages": pages
                })

    save_json(step1_data, step_dir, "1_initial_scan.json")
    print("-" * 30)

    # ---------------------------------------------------------
    # PASO 2: HEADERS Y FOOTERS (CON CONTEO DETALLADO)
    # ---------------------------------------------------------
    step_name = "2_headers_footers"
    step_dir = os.path.join(BASE_OUTPUT_DIR, step_name)
    print(f"\n[2/8] Eliminando Headers y Footers... (Salida: {step_name})")

    step2_data = []
    global_hf_log = {}

    for doc in step1_data:
        if not doc["pages"]:
            step2_data.append(doc)
            continue

        # Detectamos los patrones
        headers, footers = detect_headers_footers(doc["pages"])

        cleaned_pages = []
        file_hf_log = {}

        # Contadores específicos para este archivo
        count_headers_removed = 0
        count_footers_removed = 0

        for page in doc["pages"]:
            clean_text, removed = remove_headers_footers_with_log(page["text"], headers, footers)

            if removed:
                file_hf_log[page["page"]] = removed
                # Clasificamos qué hemos borrado para el reporte
                for line in removed:
                    if line in headers:
                        count_headers_removed += 1
                    elif line in footers:
                        count_footers_removed += 1

            new_page = page.copy()
            new_page["text"] = clean_text
            cleaned_pages.append(new_page)

        if file_hf_log:
            global_hf_log[doc["file"]] = file_hf_log
            # --- CAMBIO AQUÍ: IMPRESIÓN DETALLADA ---
            print(f"   [{doc['file']}] Eliminado: {count_headers_removed} Headers / {count_footers_removed} Footers (en {len(file_hf_log)} págs).")

        step2_data.append({
            "file": doc["file"],
            "status": doc["status"],
            "num_pages": len(cleaned_pages),
            "pages": cleaned_pages
        })

    save_json(step2_data, step_dir, "2_no_headers_footers.json")
    save_detailed_log(global_hf_log, step_dir, "log_deleted_headers_footers.txt", "REGISTRO DE HEADERS/FOOTERS ELIMINADOS")

    # ---------------------------------------------------------
    # PASO 3: NÚMEROS DE PÁGINA
    # ---------------------------------------------------------
    step_name = "3_page_numbers"
    step_dir = os.path.join(BASE_OUTPUT_DIR, step_name)
    print(f"\n[3/8] Eliminando números de página... (Salida: {step_name})")

    step3_data = []
    global_pagenum_log = {}
    docs_affected_by_pagenums = 0

    for doc in step2_data:
        if not doc["pages"]:
            step3_data.append(doc)
            continue

        cleaned_pages = []
        file_pn_log = {}
        has_deleted_pagenum = False

        for page in doc["pages"]:
            clean_text, removed = clean_page_numbers_with_log(page["text"], page["page"])
            if removed:
                file_pn_log[page["page"]] = removed
                has_deleted_pagenum = True

            new_page = page.copy()
            new_page["text"] = clean_text
            cleaned_pages.append(new_page)

        if has_deleted_pagenum:
            docs_affected_by_pagenums += 1
            global_pagenum_log[doc["file"]] = file_pn_log

        step3_data.append({
            "file": doc["file"],
            "status": doc["status"],
            "num_pages": len(cleaned_pages),
            "pages": cleaned_pages
        })

    print(f"   >>> ESTADÍSTICA: Se eliminaron números de página en {docs_affected_by_pagenums} documentos.")
    save_json(step3_data, step_dir, "3_no_page_numbers.json")
    save_detailed_log(global_pagenum_log, step_dir, "log_deleted_page_numbers.txt", "REGISTRO DE NÚMEROS DE PÁGINA ELIMINADOS")

    # ---------------------------------------------------------
    # PASO 4: ÍNDICES
    # ---------------------------------------------------------
    step_name = "4_indexes"
    step_dir = os.path.join(BASE_OUTPUT_DIR, step_name)
    print(f"\n[4/8] Detectando y eliminando Índices... (Salida: {step_name})")

    step4_data = []
    global_index_log = {}

    for doc in step3_data:
        if not doc["pages"]:
            step4_data.append(doc)
            continue

        index_indices = detect_index_pages(doc["pages"])

        if index_indices:
            print(f"   [{doc['file']}] [ÍNDICE] Detectado en páginas: {index_indices}")

            file_index_log = {}
            for p in doc["pages"]:
                if p["page"] in index_indices:
                    file_index_log[p["page"]] = p["text"].splitlines()

            global_index_log[doc["file"]] = file_index_log
            cleaned_pages = remove_index_pages(doc["pages"], index_indices)

            step4_data.append({
                "file": doc["file"],
                "status": doc["status"],
                "num_pages": len(cleaned_pages),
                "pages": cleaned_pages
            })
        else:
            step4_data.append(doc)

    save_json(step4_data, step_dir, "4_no_indexes.json")
    save_detailed_log(global_index_log, step_dir, "log_deleted_indexes.txt", "REGISTRO DE PÁGINAS DE ÍNDICE ELIMINADAS")

    # ---------------------------------------------------------
    # PASO 5: BIBLIOGRAFÍA
    # ---------------------------------------------------------
    step_name = "5_bibliography"
    step_dir = os.path.join(BASE_OUTPUT_DIR, step_name)
    print(f"\n[5/8] Detectando y eliminando Bibliografía... (Salida: {step_name})")

    step5_data = []
    global_bib_log = {}

    for doc in step4_data:
        if not doc["pages"]:
            step5_data.append(doc)
            continue

        bib_indices = detect_bib_pages(doc["pages"])

        if bib_indices:
            print(f"   [{doc['file']}] [BIBLIO] Detectada en páginas: {bib_indices}")
            cleaned_pages, file_deleted_log = clean_biblio_and_log(doc["pages"], bib_indices)

            if file_deleted_log:
                global_bib_log[doc["file"]] = file_deleted_log

            step5_data.append({
                "file": doc["file"],
                "status": doc["status"],
                "num_pages": len(cleaned_pages),
                "pages": cleaned_pages
            })
        else:
            step5_data.append(doc)

    save_json(step5_data, step_dir, "5_no_bibliography.json")
    save_detailed_log(global_bib_log, step_dir, "log_deleted_bibliography.txt", "REGISTRO DE BIBLIOGRAFÍA ELIMINADA")

    # ---------------------------------------------------------
    # PASO 6: AUTORES
    # ---------------------------------------------------------
    step_name = "6_authors"
    step_dir = os.path.join(BASE_OUTPUT_DIR, step_name)
    print(f"\n[6/8] Detectando y eliminando Autores... (Salida: {step_name})")

    step6_data = []
    global_auth_log = {}

    for doc in step5_data:
        if not doc["pages"]:
            step6_data.append(doc)
            continue

        auth_indices = detect_author_pages(doc["pages"])

        if auth_indices:
            print(f"   [{doc['file']}] [AUTORES] Detectados en páginas: {auth_indices}")
            cleaned_pages, file_deleted_log = clean_authors_and_log(doc["pages"], auth_indices)

            if file_deleted_log:
                global_auth_log[doc["file"]] = file_deleted_log

            step6_data.append({
                "file": doc["file"],
                "status": doc["status"],
                "num_pages": len(cleaned_pages),
                "pages": cleaned_pages
            })
        else:
            step6_data.append(doc)

    save_json(step6_data, step_dir, "6_no_authors.json")
    save_detailed_log(global_auth_log, step_dir, "log_deleted_authors.txt", "REGISTRO DE AUTORES ELIMINADOS")

    # ---------------------------------------------------------
    # PASO 7: REFERENCIAS PIE DE PÁGINA
    # ---------------------------------------------------------
    step_name = "7_specific_references"
    step_dir = os.path.join(BASE_OUTPUT_DIR, step_name)
    print(f"\n[7/8] Eliminando Referencias Específicas Intratexto... (Salida: {step_name})")

    step7_data = []
    global_ref_log = {}

    for doc in step6_data:
        if not doc["pages"]:
            step7_data.append(doc)
            continue

        ref_indices = detect_reference_pages(doc["pages"])

        if ref_indices:
            print(f"   [{doc['file']}] [REFERENCIAS] Detectados en páginas: {ref_indices}")
            cleaned_pages, file_deleted_log = remove_specific_references_strict(doc["pages"], ref_indices)

            if file_deleted_log:
                global_ref_log[doc["file"]] = file_deleted_log

            step7_data.append({
                "file": doc["file"],
                "status": doc["status"],
                "num_pages": len(cleaned_pages),
                "pages": cleaned_pages
            })
        else:
            step7_data.append(doc)

    save_json(step7_data, step_dir, "7_no_specific_references.json")
    save_detailed_log(global_ref_log, step_dir, "log_deleted_specific_references.txt", "REGISTRO DE REFERENCIAS ESPECÍFICAS ELIMINADAS")

    # ---------------------------------------------------------
    # PASO 8: LIMPIEZA FINAL
    # ---------------------------------------------------------
    step_name = "8_final_cleaned"
    step_dir = os.path.join(BASE_OUTPUT_DIR, step_name)
    print(f"\n[8/8] Eliminando páginas vacías y portadas... (Salida: {step_name})")

    final_data = []

    for doc in step7_data:
        if not doc["pages"]:
            final_data.append(doc)
            continue

        non_empty_pages = [p for p in doc["pages"] if len(p["text"].strip()) > 50]

        if non_empty_pages:
            final_pages = remove_front_matter(non_empty_pages, pages_to_remove=1)
        else:
            final_pages = []

        final_data.append({
            "file": doc["file"],
            "status": doc["status"],
            "num_pages": len(final_pages),
            "pages": final_pages
        })

    save_json(final_data, step_dir, "8_final_cleaned.json")

    print("\n=== PIPELINE COMPLETADO ===")
    print(f"Resultados organizados en carpetas dentro de: {BASE_OUTPUT_DIR}")

# Ejecutar
main_pipeline(EXTRACT_PATH, target_files)

=== INICIANDO PIPELINE DE PREPROCESAMIENTO CLÍNICO (ESTRUCTURADO) ===

[1/8] Escaneando PDFs... (Salida: 1_initial_scan)
--> JSON guardado: /content/drive/MyDrive/TFG_Resultados_Pipeline/1_initial_scan/1_initial_scan.json
------------------------------

[2/8] Eliminando Headers y Footers... (Salida: 2_headers_footers)
   [4084_Protocolo regional de prevención y detección de violencia en la mujer mayor de 65 años.pdf] Eliminado: 0 Headers / 47 Footers (en 47 págs).
   [15265_Protocolo Regional para la indicación, uso y autorización de dispensación de medicamentos sujetos a prescripción médica por parte de las-os enfermeras-os- ostomías.pdf] Eliminado: 49 Headers / 0 Footers (en 49 págs).
   [16384_Protocolo para la detección precoz y manejo de casos de mpox en la Región de Murcia.pdf] Eliminado: 0 Headers / 184 Footers (en 23 págs).
   [4521_Protocolo para la detección y atención de la violencia de género en atención primaria.pdf] Eliminado: 112 Headers / 56 Footers (en 56 págs).
   [45